# Fixing Portuguese-to-English Hallucinations in Whisper
## Problem: "doi-me um dente" → "diamond"

**Issue:** Whisper is phonetically hearing Portuguese correctly but choosing English words that sound similar.

**Solutions:**
1. Force Portuguese-only decoding (disable English)
2. Increase Portuguese language model weight
3. Use custom vocabulary constraints
4. Apply LLM post-processing for phonetic corrections
5. Use Portuguese-only fine-tuned models

## Setup

In [1]:
!pip install -q faster-whisper transformers openai groq
!pip install -q torch torchaudio librosa soundfile
print("✅ Installation complete")

✅ Installation complete


In [2]:
import torch
import librosa
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

🚀 Device: cuda


## Upload Audio

In [3]:
audio_path = "/content/drive/MyDrive/Gravação2.m4a"

audio, sr = librosa.load(audio_path, sr=16000, mono=True)
print(f"✅ Loaded: {audio_path} ({len(audio)/sr:.2f}s)")

display(Audio(audio, rate=sr))

✅ Loaded: /content/drive/MyDrive/Gravação2.m4a (2.97s)


---
# Solution 1: Force Portuguese-Only Decoding
---

In [4]:
print("🔧 METHOD 1: Strict Portuguese-Only Mode\n")
print("="*80)

portuguese_prompt = (
    "Esta é uma gravação em português de Portugal. "
    "Clínica Dentária Sol Nascente. "
    "Palavras comuns: dói-me, doi, dor, dente, dentes, dentista, "
    "consulta, marcação, marcar, remarcar, cancelar, "
    "urgência, emergência, dor de dentes, dor de cabeça, "
    "limpeza, obturação, ortodontia, branqueamento, coroa, "
    "ADSE, Multicare, Médis, seguro, horário, disponibilidade. "
    "Tudo em português europeu."
)

model = WhisperModel(
    "large-v3",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)

print("📊 BASELINE (Current approach):")
segments, info = model.transcribe(
    audio,
    language="pt",
    initial_prompt=portuguese_prompt,
    temperature=0.0,
    beam_size=5,
    vad_filter=True
)
baseline_result = " ".join([s.text for s in segments])
print(f"   {baseline_result}")
print()

print("📊 METHOD 1a: Temperature sampling:")
segments, info = model.transcribe(
    audio,
    language="pt",
    initial_prompt=portuguese_prompt,
    temperature=[0.0, 0.2, 0.4, 0.6],
    beam_size=5,
    vad_filter=True
)
temp_result = " ".join([s.text for s in segments])
print(f"   {temp_result}")
print()

print("📊 METHOD 1b: Larger beam size:")
segments, info = model.transcribe(
    audio,
    language="pt",
    initial_prompt=portuguese_prompt,
    temperature=0.0,
    beam_size=10,
    best_of=10,
    vad_filter=True
)
beam_result = " ".join([s.text for s in segments])
print(f"   {beam_result}")
print()

print("📊 METHOD 1c: Without VAD filter:")
segments, info = model.transcribe(
    audio,
    language="pt",
    initial_prompt=portuguese_prompt,
    temperature=0.0,
    beam_size=5,
    vad_filter=False
)
no_vad_result = " ".join([s.text for s in segments])
print(f"   {no_vad_result}")
print()

print("="*80)

🔧 METHOD 1: Strict Portuguese-Only Mode

📊 BASELINE (Current approach):
    Olá, Diamond. Posso marcar uma consulta?

📊 METHOD 1a: Temperature sampling:
    Olá, Diamond. Posso marcar uma consulta?

📊 METHOD 1b: Larger beam size:
    Olá, Diamond. Posso marcar uma consulta?

📊 METHOD 1c: Without VAD filter:
    Olá, Daimon Dietz. Posso marcar uma consulta?



---
# Solution 3: LLM Post-Processing (Recommended!)
---

This is the most effective solution: use an LLM to fix phonetic errors

In [5]:
print("🤖 METHOD 3: LLM Phonetic Correction\n")
print("This method intelligently fixes phonetic errors like 'diamond' → 'dói-me um dente'")
print()

print("Choose LLM provider:")
print("1. Groq (Free, fast - Llama 3.3 70B)")
print("2. OpenAI (GPT-4o)")
print("3. OpenAI (GPT-4o-mini) - Cheaper")

choice = input("\nEnter choice (1/2/3): ").strip()

if choice == "1":
    api_key = input("Enter Groq API key: ").strip()
    from groq import Groq
    llm_client = Groq(api_key=api_key)
    llm_model = "llama-3.3-70b-versatile"
elif choice == "2":
    api_key = input("Enter OpenAI API key: ").strip()
    from openai import OpenAI
    llm_client = OpenAI(api_key=api_key)
    llm_model = "gpt-4o"
else:
    api_key = input("Enter OpenAI API key: ").strip()
    from openai import OpenAI
    llm_client = OpenAI(api_key=api_key)
    llm_model = "gpt-4o-mini"

print(f"\n✅ Using {llm_model}\n")

🤖 METHOD 3: LLM Phonetic Correction

This method intelligently fixes phonetic errors like 'diamond' → 'dói-me um dente'

Choose LLM provider:
1. Groq (Free, fast - Llama 3.3 70B)
2. OpenAI (GPT-4o)
3. OpenAI (GPT-4o-mini) - Cheaper

✅ Using gpt-4o-mini



In [6]:
def fix_phonetic_errors(text: str) -> str:
    """Use LLM to fix phonetic transcription errors"""
    
    correction_prompt = f"""Você é um especialista em corrigir erros de transcrição automática de português europeu.

PROBLEMA COMUM:
O sistema de transcrição às vezes confunde palavras portuguesas com palavras inglesas que soam parecido:
- "diamond" deve ser "dói-me um dente" ou "doi-me o dente"
- "the" pode ser "de" ou "tê"
- "ten" pode ser "tem" ou "tém"
- "mark" pode ser "marcar"

CONTEXTO:
Esta é uma transcrição de uma chamada telefónica para a Clínica Dentária Sol Nascente em Lisboa.
O tema é: consultas dentárias, dores de dentes, marcações, urgências.

VOCABULÁRIO ESPERADO:
- Dor/sintomas: dói-me, doi, dor, dores, dor de dentes, sensibilidade, inflamação
- Anatomia: dente, dentes, gengiva, molar, canino
- Serviços: consulta, marcação, marcar, remarcar, cancelar, urgência, limpeza, obturação
- Seguros: ADSE, Multicare, Médis
- Horários: segunda, terça, quarta, quinta, sexta, sábado, manhã, tarde

TRANSCRIÇÃO ORIGINAL:
{text}

INSTRUÇÕES:
1. Identifique palavras inglesas que não fazem sentido no contexto
2. Substitua por palavras portuguesas que soam foneticamente parecidas
3. Mantenha a gramática e conjugação verbal portuguesa (pt-PT)
4. Adicione pontuação adequada
5. Corrija acentos (á, ã, ç, é, ê, í, ó, ô, ú)
6. NÃO invente informação - apenas corrija erros óbvios

EXEMPLOS:
- "I have a diamond" → "Eu tenho dor de dente" ou "dói-me um dente"
- "mark the appointment" → "marcar a consulta"
- "cancel the" → "cancelar a"

Retorne APENAS a transcrição corrigida em português europeu, sem explicações."""
    
    response = llm_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "system", "content": "Você é um especialista em português europeu e correção de transcrições."},
            {"role": "user", "content": correction_prompt}
        ],
        temperature=0.1,
        max_tokens=1000
    )
    
    return response.choices[0].message.content.strip()

print("="*80)
print("🔄 Applying LLM correction...\n")

corrected = fix_phonetic_errors(baseline_result)

print("📊 COMPARISON:")
print("="*80)
print()
print("ORIGINAL WHISPER OUTPUT:")
print(f"   {baseline_result}")
print()
print("LLM-CORRECTED OUTPUT:")
print(f"   {corrected}")
print()
print("="*80)

print("\n🔍 Phonetic fixes applied:")
if "diamond" in baseline_result.lower() and "diamond" not in corrected.lower():
    print("   ✅ Fixed: 'diamond' → (Portuguese equivalent)")
if "dente" in corrected.lower() or "doi" in corrected.lower():
    print("   ✅ Contains Portuguese dental vocabulary")
    
llm_corrected = corrected

🔄 Applying LLM correction...



AuthenticationError: Error code: 401 - {'error': {'message': "You didn't provide an API key. You need to provide your API key in an Authorization header using Bearer auth (i.e. Authorization: Bearer YOUR_KEY), or as the password field (with blank username) if you're accessing the API from your browser and are prompted for a username and password. You can obtain an API key from https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

---
# Solution 4: Multi-Model Consensus
---

In [ ]:
print("🔧 METHOD 4: Multi-Model Consensus\n")
print("="*80)

all_transcriptions = {
    "Baseline": baseline_result,
    "Temperature Sampling": temp_result,
    "Larger Beam": beam_result,
    "No VAD": no_vad_result,
}

if pt_model_result:
    all_transcriptions["PT Fine-tuned"] = pt_model_result

print("All transcriptions collected:")
for name, text in all_transcriptions.items():
    print(f"\n{name}:")
    print(f"   {text}")

print("\n" + "="*80)
print("\n🤖 Using LLM to select/merge best transcription...\n")

consensus_prompt = "Você recebeu várias transcrições da mesma chamada telefónica portuguesa.\n"
consensus_prompt += "Sua tarefa é criar a melhor transcrição possível combinando as melhores partes de cada uma.\n\n"
consensus_prompt += "CONTEXTO: Chamada para Clínica Dentária Sol Nascente (português europeu).\n"
consensus_prompt += "Tema: consultas dentárias, dores, marcações.\n\n"
consensus_prompt += "TRANSCRIÇÕES:\n"

for i, (name, text) in enumerate(all_transcriptions.items(), 1):
    consensus_prompt += f"{i}. {name}: {text}\n"

consensus_prompt += "\nINSTRUÇÕES:\n"
consensus_prompt += "1. Compare todas as transcrições\n"
consensus_prompt += "2. Identifique onde elas concordam (provavelmente correto)\n"
consensus_prompt += "3. Onde discordam, escolha a versão que faz mais sentido no contexto português\n"
consensus_prompt += "4. Elimine palavras inglesas que aparecem por erro\n"
consensus_prompt += "5. Garanta gramática portuguesa correta (pt-PT)\n"
consensus_prompt += "6. Adicione pontuação\n\n"
consensus_prompt += "Retorne a melhor transcrição final em português europeu:"

response = llm_client.chat.completions.create(
    model=llm_model,
    messages=[
        {"role": "system", "content": "Você é um especialista em português europeu."},
        {"role": "user", "content": consensus_prompt}
    ],
    temperature=0.1,
    max_tokens=1000
)

consensus_result = response.choices[0].message.content.strip()

print("📊 CONSENSUS RESULT:")
print("="*80)
print(consensus_result)
print("="*80)

---
# Final Results & Recommendations
---

In [ ]:
print("\n" + "="*80)
print("📊 FINAL RESULTS SUMMARY")
print("="*80)
print()

print("🎯 BEST APPROACHES:\n")
print("1️⃣ LLM POST-PROCESSING (Recommended):")
print(f"   {llm_corrected}")
print()
print("2️⃣ MULTI-MODEL CONSENSUS:")
print(f"   {consensus_result}")
print()

print("="*80)
print()

print("💡 RECOMMENDATIONS FOR PRODUCTION:\n")
print("1. Use Faster-Whisper Large-V3 (as you currently do)")
print("2. Apply LLM post-processing with Groq (fast + free)")
print("3. Total latency: ~2-3 seconds for transcription + 0.5-1s for correction")
print("4. Cost: Minimal (Groq is free/very cheap)")
print()
print("ALTERNATIVE:")
print("- If LLM adds too much latency, use Portuguese fine-tuned Whisper")
print("- Or use larger beam size (beam_size=10) without LLM")
print()

print("🔍 SPECIFIC FINDINGS:")
has_diamond = "diamond" in baseline_result.lower()
fixed_diamond = has_diamond and "diamond" not in llm_corrected.lower()
has_portuguese = any(word in llm_corrected.lower() for word in ["dente", "doi", "dor", "consulta"])

if has_diamond:
    print(f"   ⚠️  'diamond' detected in baseline: YES")
    if fixed_diamond:
        print(f"   ✅ LLM successfully fixed it: YES")
    else:
        print(f"   ❌ LLM fixed it: NO (may need prompt adjustment)")
        
if has_portuguese:
    print(f"   ✅ Portuguese dental vocabulary present: YES")
else:
    print(f"   ⚠️  Portuguese dental vocabulary present: NO (check audio quality)")

## Save Results

In [ ]:
import json

results = {
    "audio_file": audio_path,
    "baseline_whisper": baseline_result,
    "llm_corrected": llm_corrected,
    "multi_model_consensus": consensus_result,
    "all_variations": all_transcriptions,
    "llm_provider": llm_model
}

with open('phonetic_fix_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Results saved!")
files.download('phonetic_fix_results.json')

---
# Production Integration Code
---

Copy this code to integrate into your FastAPI app

In [ ]:
production_integration = '''
# Add this function to your main.py

from groq import Groq
import os

def fix_portuguese_transcription(text: str, groq_client: Groq) -> str:
    correction_prompt = f"""Corrija erros fonéticos nesta transcrição portuguesa de uma clínica dentária.

Erros comuns:
- Palavras inglesas que devem ser portuguesas

Vocabulário esperado: dente, dor, consulta, marcação, urgência, ADSE, Multicare.

Transcrição: {text}

Retorne apenas a versão corrigida em português europeu:"""
    
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": correction_prompt}],
            temperature=0.1,
            max_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM correction failed: {e}")
        return text

# In your transcribe endpoint:
# After getting Whisper result, add:
raw_text = result.strip()
corrected_text = fix_portuguese_transcription(raw_text, client)
return {"text": corrected_text, "raw": raw_text}
'''

print(production_integration)